In [1]:
from google.colab import drive
from IPython.display import clear_output
drive.mount('/content/drive')
!pip install -r '/content/drive/MyDrive/requirements.txt'
clear_output()

### 데이터 구성

In [ ]:
## train/val만 분리
# import os, shutil, random

# base_dir = '/content/drive/MyDrive/USC-Hackathon'
# image_train_dir = f'{base_dir}/images/train'
# image_val_dir = f'{base_dir}/images/val'
# label_train_dir = f'{base_dir}/labels/train'
# label_val_dir = f'{base_dir}/labels/val'

# # 1. 디렉토리 생성
# for d in [image_train_dir, image_val_dir, label_train_dir, label_val_dir]:
#     os.makedirs(d, exist_ok=True)

# # 2. train_country1~3에서 데이터 합치기
# all_images = []

# for country in ['train_country1', 'train_country2', 'train_country3']:
#     img_path = os.path.join(base_dir, country, 'images')
#     lbl_path = os.path.join(base_dir, country, 'labels')

#     for fname in os.listdir(img_path):
#         if not fname.endswith('.jpg'):
#             continue
#         shutil.copy(os.path.join(img_path, fname), image_train_dir)
#         label_file = fname.replace('.jpg', '.txt')
#         shutil.copy(os.path.join(lbl_path, label_file), label_train_dir)
#         all_images.append(fname)

# # 3. 일부를 val로 이동 (10%)
# random.shuffle(all_images)
# val_size = int(len(all_images) * 0.1)

# for fname in all_images[:val_size]:
#     shutil.move(os.path.join(image_train_dir, fname), image_val_dir)
#     shutil.move(os.path.join(label_train_dir, fname.replace('.jpg', '.txt')), label_val_dir)


In [ ]:
## train/val/test = 70:15:15
# import os
# import shutil
# import random

# # 경로 설정
# base_dir = '/content/drive/MyDrive/USC-Hackathon'
# image_train_dir = f'{base_dir}/images/train'
# image_val_dir   = f'{base_dir}/images/val'
# image_test_dir  = f'{base_dir}/images/test'

# label_train_dir = f'{base_dir}/labels/train'
# label_val_dir   = f'{base_dir}/labels/val'
# label_test_dir  = f'{base_dir}/labels/test'

# ##
# import shutil

# # 기존 디렉토리 삭제 (선택 사항)
# for d in [image_train_dir, image_val_dir, image_test_dir,
#           label_train_dir, label_val_dir, label_test_dir]:
#     shutil.rmtree(d, ignore_errors=True)
# ##

# # 1. 디렉토리 생성
# for d in [image_train_dir, image_val_dir, image_test_dir,
#           label_train_dir, label_val_dir, label_test_dir]:
#     os.makedirs(d, exist_ok=True)

# # 2. train_country1~3에서 전체 이미지/라벨 수집
# all_images = []

# for country in ['train_country1', 'train_country2', 'train_country3']:
#     img_path = os.path.join(base_dir, country, 'images')
#     lbl_path = os.path.join(base_dir, country, 'labels')

#     for fname in os.listdir(img_path):
#         if not fname.endswith('.jpg'):
#             continue
#         all_images.append((os.path.join(img_path, fname), os.path.join(lbl_path, fname.replace('.jpg', '.txt'))))

# # 3. 셔플 후 70:15:15 비율로 분리
# random.shuffle(all_images)
# n = len(all_images)
# train_split = int(n * 0.7)
# val_split   = int(n * 0.85)

# train_files = all_images[:train_split]
# val_files   = all_images[train_split:val_split]
# test_files  = all_images[val_split:]

# # 4. 파일 이동 함수
# def move_files(file_list, image_dir, label_dir):
#     for img_path, lbl_path in file_list:
#         shutil.copy(img_path, os.path.join(image_dir, os.path.basename(img_path)))
#         shutil.copy(lbl_path, os.path.join(label_dir, os.path.basename(lbl_path)))

# # 5. 각 세트로 이동
# move_files(train_files, image_train_dir, label_train_dir)
# move_files(val_files, image_val_dir, label_val_dir)
# move_files(test_files, image_test_dir, label_test_dir)

# print(f"분할 완료: 총 {n}개")
# print(f"  ▶ Train: {len(train_files)}개")
# print(f"  ▶ Val  : {len(val_files)}개")
# print(f"  ▶ Test : {len(test_files)}개")


분할 완료: 총 6039개
  ▶ Train: 4227개
  ▶ Val  : 906개
  ▶ Test : 906개


In [5]:
# import os
# import cv2
# import albumentations as A
# from glob import glob
# from tqdm import tqdm

# # 경로
# image_dir = '/content/drive/MyDrive/USC-Hackathon/project/images/train'
# label_dir = '/content/drive/MyDrive/USC-Hackathon/project/labels/train'

# aug_image_dir = '/content/drive/MyDrive/USC-Hackathon/project/images/train_aug'
# aug_label_dir = '/content/drive/MyDrive/USC-Hackathon/project/labels/train_aug'

# os.makedirs(aug_image_dir, exist_ok=True)
# os.makedirs(aug_label_dir, exist_ok=True)

# # 증강 대상 클래스: 부족 클래스
# target_class_ids = [0, 1]  # Pothole, Alligator Crack

# # 증강 파이프라인
# transform = A.Compose([
#     A.HorizontalFlip(p=0.5),
#     A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.7),
#     A.MotionBlur(blur_limit=3, p=0.3),
#     A.GaussNoise(p=0.3),
#     A.CLAHE(p=0.4),
#     A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=10, val_shift_limit=10, p=0.4),
#     A.Rotate(limit=10, border_mode=cv2.BORDER_CONSTANT, p=0.4),
#     A.Affine(scale=(0.9, 1.1), translate_percent=0.1, p=0.5),
#     A.CoarseDropout(max_holes=4, max_height=30, max_width=30, p=0.3)
# ], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

# # 라벨 파일 읽기
# def read_yolo_label(txt_path):
#     boxes, class_labels = [], []
#     with open(txt_path, 'r') as f:
#         for line in f.readlines():
#             parts = line.strip().split()
#             cls_id = int(parts[0])
#             if cls_id in target_class_ids:
#                 box = list(map(float, parts[1:]))
#                 boxes.append(box)
#                 class_labels.append(cls_id)
#     return boxes, class_labels

# # 증강 실행
# aug_count = 0
# for img_path in tqdm(glob(f'{image_dir}/*.jpg')):
#     basename = os.path.basename(img_path).replace('.jpg', '')
#     label_path = os.path.join(label_dir, f'{basename}.txt')

#     if not os.path.exists(label_path):
#         continue

#     bboxes, class_labels = read_yolo_label(label_path)
#     if len(bboxes) == 0:
#         continue  # 증강 대상 클래스 없음

#     img = cv2.imread(img_path)
#     if img is None:
#         continue  # 이미지 읽기 실패

#     try:
#         for i in range(3):  # 증강 반복 수 (조정 가능)
#             augmented = transform(image=img, bboxes=bboxes, class_labels=class_labels)
#             aug_img = augmented['image']
#             aug_bboxes = augmented['bboxes']
#             aug_classes = augmented['class_labels']

#             aug_filename = f'{basename}_aug{aug_count}'
#             cv2.imwrite(os.path.join(aug_image_dir, f'{aug_filename}.jpg'), aug_img)

#             with open(os.path.join(aug_label_dir, f'{aug_filename}.txt'), 'w') as f:
#                 for cls, bbox in zip(aug_classes, aug_bboxes):
#                     f.write(f"{cls} {' '.join(map(str, bbox))}\n")

#             aug_count += 1

#     except Exception as e:
#         print(f"오류 발생: {img_path} - {e}")
#         continue

# print(f"✅ 총 증강 이미지 수: {aug_count}")


/tmp/ipython-input-5-3355248322.py:30: UserWarning: Argument(s) 'max_holes, max_height, max_width' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=4, max_height=30, max_width=30, p=0.3)
100%|██████████| 4227/4227 [39:40<00:00,  1.78it/s]

✅ 총 증강 이미지 수: 8223


In [3]:
# import shutil
# import os
# import cv2
# from glob import glob
# from tqdm import tqdm

# # 경로 설정
# image_dir = '/content/drive/MyDrive/USC-Hackathon/project/images/train'
# label_dir = '/content/drive/MyDrive/USC-Hackathon/project/labels/train'
# aug_image_dir = '/content/drive/MyDrive/USC-Hackathon/project/images/train_aug'
# aug_label_dir = '/content/drive/MyDrive/USC-Hackathon/project/labels/train_aug'

# # 이미지 파일 복사
# aug_image_files = glob(f'{aug_image_dir}/*.jpg')
# image_remaining = [f for f in aug_image_files if not os.path.exists(os.path.join(image_dir, os.path.basename(f)))]

# print(f"복사할 증강 이미지 수: {len(image_remaining)}")
# for f in tqdm(image_remaining, desc="이미지 복사 진행 중"):
#     dst = os.path.join(image_dir, os.path.basename(f))
#     shutil.copy(f, dst)

# # 라벨 파일 복사
# aug_label_files = glob(f'{aug_label_dir}/*.txt')
# label_remaining = [f for f in aug_label_files if not os.path.exists(os.path.join(label_dir, os.path.basename(f)))]

# print(f"복사할 증강 라벨 수: {len(label_remaining)}")
# for f in tqdm(label_remaining, desc="라벨 복사 진행 중"):
#     dst = os.path.join(label_dir, os.path.basename(f))
#     shutil.copy(f, dst)

# print("✅ 증강 데이터 복사 완료")

복사할 증강 이미지 수: 0


이미지 복사 진행 중: 0it [00:00, ?it/s]


복사할 증강 라벨 수: 4010


라벨 복사 진행 중: 100%|██████████| 4010/4010 [03:12<00:00, 20.80it/s] 

✅ 증강 데이터 복사 완료


In [2]:
import os
import cv2
import albumentations as A
from glob import glob
from tqdm import tqdm

# 원본 데이터 경로
image_dir = '/content/drive/MyDrive/USC-Hackathon/project/images/train'
label_dir = '/content/drive/MyDrive/USC-Hackathon/project/labels/train'

# 증강된 데이터 저장 경로
aug_image_dir = '/content/drive/MyDrive/USC-Hackathon/project/images/train_aug'
aug_label_dir = '/content/drive/MyDrive/USC-Hackathon/project/labels/train_aug'

os.makedirs(aug_image_dir, exist_ok=True)
os.makedirs(aug_label_dir, exist_ok=True)

# 증강 대상 클래스: 0, 1번 클래스 중심
target_class_ids = [0, 1]

# 증강 파이프라인
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.4),
    A.MotionBlur(blur_limit=3, p=0.3),
    A.GaussNoise(p=0.3),
    A.CLAHE(p=0.3),
    A.HueSaturationValue(p=0.3),
    A.Affine(scale=(0.9, 1.1), translate_percent=0.1, p=0.4),
    A.Rotate(limit=10, p=0.3),
    A.CoarseDropout(max_height=30, max_width=30, p=0.3),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

# YOLO 형식 라벨 읽기 함수
def read_yolo_label(txt_path):
    with open(txt_path, 'r') as f:
        lines = f.readlines()
    boxes, class_labels = [], []
    for line in lines:
        parts = line.strip().split()
        cls_id = int(float(parts[0]))
        if cls_id in target_class_ids:
            boxes.append(list(map(float, parts[1:])))
            class_labels.append(cls_id)
    return boxes, class_labels

# 증강 실행
aug_count = 0
all_images = glob(f'{image_dir}/*.jpg')

for img_path in tqdm(all_images, desc="증강 중"):
    basename = os.path.basename(img_path).replace('.jpg', '')
    label_path = os.path.join(label_dir, f'{basename}.txt')

    if not os.path.exists(label_path):
        continue

    bboxes, class_labels = read_yolo_label(label_path)
    if len(bboxes) == 0:
        continue

    img = cv2.imread(img_path)

    try:
        for i in range(3):  # 이미지당 증강 반복 횟수
            augmented = transform(image=img, bboxes=bboxes, class_labels=class_labels)
            aug_img = augmented['image']
            aug_bboxes = augmented['bboxes']
            aug_classes = augmented['class_labels']

            aug_filename = f'{basename}_aug{aug_count}'
            cv2.imwrite(os.path.join(aug_image_dir, f'{aug_filename}.jpg'), aug_img)

            with open(os.path.join(aug_label_dir, f'{aug_filename}.txt'), 'w') as f:
                for cls, bbox in zip(aug_classes, aug_bboxes):
                    f.write(f"{cls} {' '.join(map(str, bbox))}\n")

            aug_count += 1

    except Exception as e:
        print(f"⚠️ 오류 발생: {img_path} - {e}")
        continue

print(f"✅ 총 증강 이미지 수: {aug_count}")


/tmp/ipython-input-2-2151461881.py:31: UserWarning: Argument(s) 'max_height, max_width' are not valid for transform CoarseDropout
  A.CoarseDropout(max_height=30, max_width=30, p=0.3),
증강 중: 100%|██████████| 5436/5436 [40:54<00:00,  2.21it/s]

✅ 총 증강 이미지 수: 10512


In [4]:
import os
import shutil
from tqdm import tqdm

# 경로 설정
base_dir = '/content/drive/MyDrive/USC-Hackathon/project'
image_train_dir = os.path.join(base_dir, 'images/train')
label_train_dir = os.path.join(base_dir, 'labels/train')
image_aug_dir = os.path.join(base_dir, 'images/train_aug')
label_aug_dir = os.path.join(base_dir, 'labels/train_aug')

# 파일 이름 기준으로 중복 체크
existing_files = set(os.path.splitext(f)[0] for f in os.listdir(image_train_dir) if f.endswith('.jpg'))

added_count = 0
skipped_count = 0

for img_file in tqdm(os.listdir(image_aug_dir), desc="train_aug → train 병합 중"):
    if not img_file.endswith('.jpg'):
        continue
    base_name = os.path.splitext(img_file)[0]

    if base_name in existing_files:
        skipped_count += 1
        continue  # 중복이면 skip

    # 이미지와 라벨 경로
    src_img = os.path.join(image_aug_dir, f'{base_name}.jpg')
    src_lbl = os.path.join(label_aug_dir, f'{base_name}.txt')
    dst_img = os.path.join(image_train_dir, f'{base_name}.jpg')
    dst_lbl = os.path.join(label_train_dir, f'{base_name}.txt')

    # 이미지 복사
    shutil.copy2(src_img, dst_img)
    # 라벨이 존재할 경우에만 복사
    if os.path.exists(src_lbl):
        shutil.copy2(src_lbl, dst_lbl)

    added_count += 1

print(f"\n✅ 병합 완료: {added_count}개 추가, {skipped_count}개 중복으로 제외")


train_aug → train 병합 중: 100%|██████████| 10512/10512 [00:00<00:00, 339113.53it/s]


✅ 병합 완료: 0개 추가, 10512개 중복으로 제외


In [5]:
import os

label_dir = '/content/drive/MyDrive/USC-Hackathon/project/labels/train'

# 수정 대상 파일만 필터링 (파일명에 'aug' 포함)
label_files = [f for f in os.listdir(label_dir) if f.endswith('.txt') and 'aug' in f]

modified_count = 0

for file in label_files:
    file_path = os.path.join(label_dir, file)
    with open(file_path, 'r') as f:
        lines = f.readlines()

    new_lines = []
    changed = False

    for line in lines:
        parts = line.strip().split()
        if not parts:
            continue
        try:
            class_id = int(float(parts[0]))  # float → int 변환
            rest = parts[1:]
            new_line = f"{class_id} {' '.join(rest)}\n"
            new_lines.append(new_line)
            if parts[0] != str(class_id):  # 기존값이 int가 아니면 변경된 것
                changed = True
        except:
            print(f"⚠️ 오류: {file} → {line.strip()}")

    if changed:
        with open(file_path, 'w') as f:
            f.writelines(new_lines)
        modified_count += 1

print(f"✅ 클래스 ID를 int로 수정한 파일 수: {modified_count}개")


✅ 클래스 ID를 int로 수정한 파일 수: 10430개
